<a href="https://colab.research.google.com/github/rsekola/Projet_data/blob/master/Apprentissage_automatique_apprentissage_supervis%C3%A9%2C_r%C3%A9gression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import io

# Load data
df = pd.read_csv('5G_energy_consumption_dataset.csv')

# Basic info
info_buf = io.StringIO()
df.info(buf=info_buf)
info_text = info_buf.getvalue()

# Summary stats
describe_text = df.describe(include='all').to_string()

# Missing values
missing = df.isna().sum()

# Duplicates
duplicate_count = df.duplicated().sum()

# Handle duplicates (remove for demonstration)
df_no_dup = df.drop_duplicates()

# Basic outlier detection using IQR for numeric cols
numeric_cols = df_no_dup.select_dtypes(include='number').columns
outliers = {}
for col in numeric_cols:
    q1 = df_no_dup[col].quantile(0.25)
    q3 = df_no_dup[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers[col] = df_no_dup[(df_no_dup[col] < lower) | (df_no_dup[col] > upper)][col].count()

# Categorical encoding: simple label encoding demonstration
from sklearn.preprocessing import LabelEncoder
encoded_df = df_no_dup.copy()
label_encoders = {}
for col in encoded_df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    encoded_df[col] = le.fit_transform(encoded_df[col].astype(str))
    label_encoders[col] = le.classes_

# Return key outputs
{
    "info": info_text,
    "describe": describe_text,
    "missing_values": missing.to_dict(),
    "duplicate_count": duplicate_count,
    "outliers_count": outliers,
    "categorical_columns_encoded": list(label_encoders.keys()),
    "label_classes": {col: classes.tolist() for col, classes in label_encoders.items()}
}



{'info': "<class 'pandas.core.frame.DataFrame'>\nRangeIndex: 92629 entries, 0 to 92628\nData columns (total 6 columns):\n #   Column   Non-Null Count  Dtype  \n---  ------   --------------  -----  \n 0   Time     92629 non-null  object \n 1   BS       92629 non-null  object \n 2   Energy   92629 non-null  float64\n 3   load     92629 non-null  float64\n 4   ESMODE   92629 non-null  float64\n 5   TXpower  92629 non-null  float64\ndtypes: float64(4), object(2)\nmemory usage: 4.2+ MB\n",
 'describe': '                   Time     BS        Energy          load        ESMODE       TXpower\ncount             92629  92629  92629.000000  92629.000000  92629.000000  92629.000000\nunique              168    923           NaN           NaN           NaN           NaN\ntop     20230102 230000  B_583           NaN           NaN           NaN           NaN\nfreq                766    130           NaN           NaN           NaN           NaN\nmean                NaN    NaN     28.138997      0.2447

In [3]:
df.columns


Index(['Time', 'BS', 'Energy', 'load', 'ESMODE', 'TXpower'], dtype='object')

In [7]:
from sklearn.model_selection import train_test_split

# Indiquez ici votre variable cible :
target_column = "TXpower"

# Séparation features / target
X = encoded_df.drop(columns=[target_column])
y = encoded_df[target_column]

# Division Train/Test (80% - 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape


((74103, 5), (18526, 5), (74103,), (18526,))

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

#Indiquez ici votre variable cible
target_column = "TXpower"

# Séparation des variables explicatives et de la cible
X = encoded_df.drop(columns=[target_column])
y = encoded_df[target_column]

# Division en train / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Modèle de régression
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    max_depth=None,
    min_samples_split=2
)

# Entraînement
model.fit(X_train, y_train)

# Prédictions
y_pred = model.predict(X_test)

# Évaluation du modèle
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

rmse, r2


(np.float64(0.06507207752889767), 0.9554925976197002)

In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Prédictions sur l'ensemble de test
y_pred = model.predict(X_test)

# Calcul des métriques
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Affichage
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R²: {r2:.4f}")


RMSE: 0.0651
MAE: 0.0256
R²: 0.9555
